In [ ]:
import os
import csv
import argparse
import numpy as np
import scipy.io as sio
from natsort import natsorted
import codecs
import re

In [32]:
# -------------------------------------------------
# 解析 label txt
# -------------------------------------------------
def parse_label_txt(label_path):

    info = {
        "class": None,
        "channel": None,
        "intensity": None,
        "env": ""
    }

    try:
        with codecs.open(label_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                # 前三行都有 =
                if "=" in line:

                    key, val = line.split("=", 1)

                    key = key.lower()

                    if "data_path" in key:
                        info["class"] = val.strip()

                    elif "channel" in key:
                        info["channel"] = int(val)

                    elif "intensity" in key:
                        info["intensity"] = int(val)

                # 第四行（环境描述）
                else:
                    parts = line.split()
                    info["env"] = parts[-1]

    except Exception as e:
        print("读取label失败:", label_path, e)

    return info

# -------------------------------------------------
# 找 label
# -------------------------------------------------
def find_label_for_event_and_parse(label_root, class_name, session_folder):

    candidate_dir = os.path.join(label_root, class_name)

    if not os.path.isdir(candidate_dir):
        return None, None

    for fn in os.listdir(candidate_dir):

        name, ext = os.path.splitext(fn)

        s1 = re.sub(r'[^0-9]', '', session_folder)
        s2 = re.sub(r'[^0-9]', '', name)

        if s1 in s2 or s2 in s1:

            full = os.path.join(candidate_dir, fn)

            if ext.lower() == ".txt":

                parsed = parse_label_txt(full)

                return full, parsed

    return None, None

# -------------------------------------------------
# 读取 mat
# -------------------------------------------------
def load_and_combine_mat_files(session_path):

    mat_files = [f for f in os.listdir(session_path) if f.endswith(".mat")]

    mat_files = natsorted(mat_files)

    arrays = []

    for f in mat_files:

        file_path = os.path.join(session_path, f)

        try:

            mat = sio.loadmat(file_path)

            if "s" not in mat:
                continue

            arr = mat["s"]

            arrays.append(arr)

        except Exception as e:
            print("读取失败:", file_path, e)

    if not arrays:
        return None

    combined = np.concatenate(arrays, axis=1)

    return combined

# -------------------------------------------------
# 保存窗口
# -------------------------------------------------
def save_windows_and_metadata(
        combined_arr,
        out_dir,
        class_name,
        session_folder,
        window_len,
        hop,
        label_parsed,
        meta_writer
):

    ch, total = combined_arr.shape

    os.makedirs(out_dir, exist_ok=True)

    idx = 0

    for start in range(0, total - window_len + 1, hop):

        win = combined_arr[:, start:start + window_len]

        save_arr = win.T.astype(np.float32)

        fname = f"{class_name}__{session_folder}__{idx:06d}.npy"

        path = os.path.join(out_dir, fname)

        np.save(path, save_arr)

        cls = label_parsed.get("class") if label_parsed else ""

        chn = label_parsed.get("channel") if label_parsed else ""

        inten = label_parsed.get("intensity") if label_parsed else ""

        env = label_parsed.get("env") if label_parsed else ""

        meta_writer.writerow([
            fname,
            class_name,
            session_folder,
            start,
            start + window_len,
            cls,
            chn,
            inten,
            env
        ])

        idx += 1

    return idx
    # -------------------------------------------------
# 主函数
# -------------------------------------------------
def main(args):

    data_root = args.data_root
    label_root = args.label_root
    out_root = args.out_root

    window_len = args.window_len
    hop = args.hop

    os.makedirs(out_root, exist_ok=True)

    metadata_path = os.path.join(out_root, "metadata.csv")

    with open(metadata_path, "w", newline='', encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "filename",
            "class",
            "session_folder",
            "start",
            "end",
            "label_class",
            "label_channel",
            "label_intensity",
            "label_env"
        ])

        for class_name in os.listdir(data_root):

            class_dir = os.path.join(data_root, class_name)

            if not os.path.isdir(class_dir):
                continue

            print("处理类别:", class_name)

            out_class_dir = os.path.join(out_root, class_name)

            for session_folder in os.listdir(class_dir):

                session_path = os.path.join(class_dir, session_folder)

                if not os.path.isdir(session_path):
                    continue

                print("  session:", session_folder)

                combined = load_and_combine_mat_files(session_path)

                if combined is None:
                    print("  无有效数据")
                    continue

                label_path, label_parsed = find_label_for_event_and_parse(
                    label_root,
                    class_name,
                    session_folder
                )

                save_windows_and_metadata(
                    combined,
                    out_class_dir,
                    class_name,
                    session_folder,
                    window_len,
                    hop,
                    label_parsed,
                    writer
                )

    print("预处理完成")
    print("metadata:", metadata_path)

# -------------------------------------------------
# CLI
# -------------------------------------------------
# if __name__ == "__main__":

#     parser = argparse.ArgumentParser()

#     parser.add_argument("--data_root", required=True)
#     parser.add_argument("--label_root", required=True)
#     parser.add_argument("--out_root", required=True)

#     parser.add_argument("--window_len", type=int, default=100)
#     parser.add_argument("--hop", type=int, default=50)

#     args = parser.parse_args()

#     main(args)

In [33]:
args = argparse.Namespace(
            data_root="/home/sente/das_ai_project/data/train/data",
            label_root="/home/sente/das_ai_project/data/train/label",
            out_root="/home/sente/das_ai_project/data/train/processed_data",
            window_len=100,
            hop=50
)
main(args)

处理类别: pawang
  session: 2021-10-22 14_41_03
  session: 2021-10-22 15_12_21
  session: 2021-10-22 14_51_38
  session: 2021-10-22 14_37_20
  session: 2021-10-22 14_58_22
  session: 2021-10-22 15_01_31
  session: 2021-10-22 14_47_59
  session: 2021-10-22 14_54_57
  session: 2021-10-22 15_04_31
  session: 2021-10-22 14_44_37
处理类别: zuanwang
  session: 2021-10-22 14_10_07
  session: 2021-10-22 13_43_20
  session: 2021-10-22 14_13_42
  session: 2021-10-22 13_53_03
  session: 2021-10-22 14_16_34
  session: 2021-10-22 14_02_39
  session: 2021-10-22 13_55_58
  session: 2021-10-22 13_45_48
  session: 2021-10-22 13_59_01
  session: 2021-10-22 14_06_07
预处理完成
metadata: /home/sente/das_ai_project/data/train/processed_data/metadata.csv
